In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 50)

In [0]:
df = spark.read.format("csv").option("header", True).load("/Volumes/external-catalog/default/test-volume/Employee_Attrition.csv")
display(df)

In [0]:
# Filter for high risk attrition employees
high_risk_df = df.filter((df["Attrition"] == "No") & (df["JobSatisfaction"] < 3))

# Select relevant columns
selected_df = high_risk_df.select("EmployeeNumber", "Department", "JobRole", "JobSatisfaction", "Attrition")

# Write to Delta table in default schema of external-catalog
selected_df.write.format("delta").mode("overwrite").saveAsTable("`external-catalog`.`default`.high_risk_attrition_employees")

In [0]:
df1 = spark.table("`external-catalog`.`default`.high_risk_attrition_employees")
display(df1)

In [0]:
history_df = spark.sql("DESCRIBE HISTORY `external-catalog`.`default`.high_risk_attrition_employees")
display(history_df)


In [0]:
from pyspark.sql import Row

# Create a DataFrame with a dummy record
dummy_data = [Row(EmployeeNumber="99999", Department="DummyDept", JobRole="DummyRole", JobSatisfaction="1", Attrition="No")]
dummy_df = spark.createDataFrame(dummy_data)

# Insert the dummy record into the Delta table
dummy_df.write.format("delta").mode("append").saveAsTable("`external-catalog`.`default`.high_risk_attrition_employees")

In [0]:
df_delta = spark.read.format("delta").table("`external-catalog`.`default`.high_risk_attrition_employees")
display(df_delta)

In [0]:
spark.sql("""
  UPDATE `external-catalog`.`default`.high_risk_attrition_employees
  SET Department = 'Sale'
  WHERE EmployeeNumber = '99999'
""");

In [0]:
history_df = spark.sql("DESCRIBE HISTORY `external-catalog`.`default`.high_risk_attrition_employees")
display(history_df.select("version","operation","timestamp"))

In [0]:
#read delta table from speific version
df_version = spark.read.format("delta").option("versionAsOf",0).table("`external-catalog`.`default`.high_risk_attrition_employees")
display(df_version)

In [0]:
#timetravel - read delta table from specific timestamp
df_timestamp = spark.read.format("delta").option("timestampAsOf", "2026-01-08T05:09:22.806+00:00").table("`external-catalog`.`default`.high_risk_attrition_employees")
display(df_timestamp)

In [0]:
#create volume in catalog 
spark.sql("CREATE VOLUME `external-catalog`.`default`.employee_transformed_data")

In [0]:
#apply some logical transformation on employee dataframe and write it into volume partitioned by department
# Logical transformation: filter employees with JobSatisfaction < 3 and Attrition == 'No'
from pyspark.sql.functions import col,upper

transformed_df = df.withColumn("JobRole",upper(col("JobRole")))
# Write to volume partitioned by Department
transformed_df.write.partitionBy("Department").mode("overwrite").format("parquet").save("/Volumes/external-catalog/default/employee_transformed_data")